# Build a BGP Knowledge Graph — in Python

A tiny, **runnable** companion to the *Build a Knowledge Graph from Scratch* course —
same idea as the OSPF notebook, different protocol. **Python + networkx**, so it runs
right here in Colab. No database to install.

**The one idea:** BGP is already a graph — routers are nodes, sessions (peerings) are
edges, best-path selection is a traversal. A *knowledge graph* is that same machine,
except **you** pick the nodes — so you can add the one node BGP never had: the
**business service** that dies when a prefix does.

**Is this machine learning?** No. It is just structured facts you can question — no
training, no guessing. If you can read a BGP table, you can read this.


## The words you need

- **node** — a thing. `edge-rtr-01`, `203.0.113.0/24`, `Customer Portal`.
- **edge** — a named, directed link. `edge-rtr-01 --PEERS_WITH--> upstream-isp-01`.
- **label** — a node's type. `Router`, `Prefix`, `Service`.
- **property** — a fact stored on a node or edge. `state='down'`, `type='eBGP'`.
- **AS / ASN** — one administrative routing domain, identified by a number.
- **eBGP / iBGP** — a session that crosses an AS boundary, vs. one that stays inside it.
- **traversal** — walking the links to answer a question (that is the blast radius).


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# networkx and matplotlib come pre-installed in Colab.
# A DiGraph is a *directed* graph: every edge has a direction, like our arrows.
G = nx.DiGraph()
print('empty graph ready:', G)

## Step 1 — Create your first nodes

Each node gets a **label** (its type) and some **properties** (facts about it).
We make three routers: our edge router, our upstream transit provider, and one
internal core router.


In [ ]:
# our edge router, our upstream transit, and one internal core router
G.add_node('edge-rtr-01',     label='Router', asn=65001, role='edge')
G.add_node('upstream-isp-01', label='Router', asn=701,   role='transit')
G.add_node('core-rtr-01',     label='Router', asn=65001, role='core')

print(G.number_of_nodes(), 'nodes:', list(G.nodes))

## Step 2 — Connect them (the session lives on the edge)

Just like OSPF's `CONNECTS` link carried `state='down'` right on the edge, a BGP
**session** doesn't need its own node — it *is* the edge between two routers, and
its `type` (eBGP/iBGP) and `state` (up/down) live on it.

`edge-rtr-01` peers eBGP with `upstream-isp-01` — and that session is **down**. It
also peers iBGP with `core-rtr-01`, which is still up.


In [ ]:
G.add_edge('edge-rtr-01', 'upstream-isp-01', rel='PEERS_WITH', type='eBGP', state='down')  # <-- the failure
G.add_edge('edge-rtr-01', 'core-rtr-01',     rel='PEERS_WITH', type='iBGP', state='up')

for u, v, d in G.edges(data=True):
    print(f'{u} --{d["rel"]}({d["type"]}, {d["state"]})--> {v}')

## Step 3 — Tell the BGP story

`edge-rtr-01` **advertises** a prefix out — specifically *via* its session to
`upstream-isp-01` (that's the `via` property: it ties a prefix to the one session
it depends on). A business service then **rides on** that prefix. This is the link
no `show ip bgp` command holds: a routing fact wired to a business fact.


In [ ]:
G.add_node('203.0.113.0/24', label='Prefix')
G.add_node('Customer Portal', label='Service', criticality='critical')

G.add_edge('edge-rtr-01', '203.0.113.0/24', rel='ADVERTISES', via='upstream-isp-01')
G.add_edge('203.0.113.0/24', 'Customer Portal', rel='SUPPORTS')

print('graph now has', G.number_of_nodes(), 'nodes and', G.number_of_edges(), 'edges')

## See it

Colour the nodes by their label; draw the down eBGP session red and dashed, and the
up iBGP session in the normal colour.


In [ ]:
colors = {'Router': '#3aa0ff', 'Prefix': '#0f7f74', 'Service': '#c8702a'}
node_colors = [colors.get(G.nodes[n].get('label'), '#cccccc') for n in G]
pos = nx.spring_layout(G, seed=7)   # seed keeps the layout stable

plt.figure(figsize=(9, 6))
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1800, alpha=0.92)
nx.draw_networkx_labels(G, pos, font_size=8)

down  = [(u, v) for u, v, d in G.edges(data=True) if d.get('state') == 'down']
other = [(u, v) for u, v, d in G.edges(data=True) if (u, v) not in down]
nx.draw_networkx_edges(G, pos, edgelist=other, arrows=True, arrowsize=16, edge_color='#8894a0')
nx.draw_networkx_edges(G, pos, edgelist=down,  arrows=True, arrowsize=16, edge_color='#d34b3f', style='dashed')
nx.draw_networkx_edge_labels(G, pos, font_size=7,
    edge_labels={(u, v): d['rel'] for u, v, d in G.edges(data=True)})

plt.axis('off'); plt.title('BGP knowledge graph'); plt.tight_layout(); plt.show()

## Step 4 — Ask it the 3 AM question

*When our eBGP session to upstream-isp-01 is down, which services lose reachability?*

We **traverse**: a down eBGP session -> its neighbor's name -> the prefixes this
router tagged as depending on that neighbor (`via`) -> the services those prefixes
support. The iBGP session to `core-rtr-01` never enters this walk — it's still up,
and nothing is tagged `via` it.


In [ ]:
def blast_radius(G):
    """Services put at risk by any eBGP session that is down."""
    at_risk = []
    for rtr, nbr, d in G.edges(data=True):
        # 1) an eBGP session that is down
        if d.get('rel') == 'PEERS_WITH' and d.get('type') == 'eBGP' and d.get('state') == 'down':
            # 2) prefixes this router advertises specifically via that neighbor
            for _, pfx, d2 in G.out_edges(rtr, data=True):
                if d2.get('rel') != 'ADVERTISES' or d2.get('via') != nbr:
                    continue
                # 3) services those prefixes support
                for _, svc, d3 in G.out_edges(pfx, data=True):
                    if d3.get('rel') == 'SUPPORTS':
                        at_risk.append((rtr, nbr, pfx, svc))
    return at_risk

for rtr, nbr, pfx, svc in blast_radius(G):
    print(f'{rtr} --eBGP down--> {nbr}   {pfx}  ->  AT RISK: {svc}')

The router only ever told you a session dropped. Your graph just told you the
**Customer Portal** is at risk — because you gave it the one node BGP never had, and
tagged exactly which session each prefix depends on.


## What-if: repair the session, then break it again

Flip the eBGP session to `up` and ask again — nothing is at risk (the iBGP session
was never the problem). Set it back to `down` — the Customer Portal returns. You are
running "what if this fails?" on a model, safely.


In [ ]:
G['edge-rtr-01']['upstream-isp-01']['state'] = 'up'      # repair
print('after repair: ', blast_radius(G) or 'nothing at risk')

G['edge-rtr-01']['upstream-isp-01']['state'] = 'down'    # break it again
print('after re-break:', [svc for *_, svc in blast_radius(G)])

## Make it yours

Change the five values below to **one** eBGP session, prefix, and service *you* run,
then re-run. Your own service name comes back from `blast_radius(G)` — that is the
moment the tool is yours. (Model the smallest slice that answers one real question;
you can always add one more node later.)


In [ ]:
# --- change these five values to your network ---
MY_ROUTER   = 'edge-rtr-02'
MY_UPSTREAM = 'isp-secondary'
MY_ASN      = 65099
MY_PREFIX   = '198.51.100.0/24'
MY_SERVICE  = 'Payments API'
# --------------------------------------------------

G.add_node(MY_ROUTER,   label='Router', asn=MY_ASN, role='edge')
G.add_node(MY_UPSTREAM, label='Router', asn=64999,  role='transit')
G.add_node(MY_PREFIX,   label='Prefix')
G.add_node(MY_SERVICE,  label='Service', criticality='critical')

G.add_edge(MY_ROUTER, MY_UPSTREAM, rel='PEERS_WITH', type='eBGP', state='down')  # the modelled failure
G.add_edge(MY_ROUTER, MY_PREFIX,   rel='ADVERTISES', via=MY_UPSTREAM)
G.add_edge(MY_PREFIX, MY_SERVICE,  rel='SUPPORTS')

for rtr, nbr, pfx, svc in blast_radius(G):
    print(f'{rtr} --eBGP down--> {nbr}   AT RISK: {svc}')

## You built a BGP knowledge graph

From an empty graph to a question no `show ip bgp summary` can answer. The shapes
you used — *a service depends on a prefix, a prefix is advertised via a specific
session, a router peers with another router* — are the same ones every bigger graph
is made of, and the same shapes you already used for OSPF.

**Where next:** the full course explains the *why* in depth, and the companion lab
does this exact thing in **Neo4j + Cypher** (a real graph database) instead of
networkx — same nodes, same edges, same blast-radius question. From there, the
natural next step is wiring an OSPF `NextHop` into a BGP `Prefix` — the moment your
graph spans two protocols at once.
